# Feature Engineering — DataCo Smart Supply Chain for Data Analysis

**Project phase:** Feature Engineering (CRISP-DM Phase 3 — Data Preparation)
**Input:** `DataCoSupplyChain_Cleaned.csv` (output of the Data Cleaning notebook)
**Analyst:** Akash More

This notebook turns the cleaned dataset into a feature set that supports three problems:

1. **Late delivery classification** — will an order arrive late?
2. **Order profitability regression** — how profitable is an order likely to be?
3. **Customer, product, and regional analysis** — spend, frequency, demand, and risk, at the level a
   supply-chain or e-commerce team would actually report on.

Fourteen feature groups are built below, in the order a stakeholder would ask for them: time, delivery,
sales, profit, customer, product, shipping, regional, risk, and behavioral features, followed by a KPI
rollup, the final ML-ready feature table, and a validation pass before export.

**A note on leakage.** Several features here are full-history aggregates (e.g. a customer's total lifetime
spend, a region's overall late-delivery rate). Those are exactly right for a dashboard or a segmentation
report — they're wrong for a model that has to predict at order time, because the model wouldn't know a
customer's *future* orders yet. Every such feature is labeled below, and a point-in-time-safe alternative is
provided for the two prediction targets (`is_late`, `profit_margin`).

**A note on scope.** The dataset has no inventory table (stock levels, reorder points), so inventory
features aren't buildable from this data — that section is intentionally left out. It also has no direct
shipping-cost column, so shipping-cost features are replaced with delay/efficiency proxies built from the
columns that do exist.

## 1. Load Cleaned Data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("supply_chain_orders_cleaned.csv", encoding="latin1")
df["order date (DateOrders)"] = pd.to_datetime(df["order date (DateOrders)"])
df["shipping date (DateOrders)"] = pd.to_datetime(df["shipping date (DateOrders)"])

df_fe = df.copy()
print(f"Starting shape: {df_fe.shape}")
df_fe.head()

Starting shape: (180519, 48)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order State,Order Status,Product Card Id,Product Category Id,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode,is_profit_outlier
0,DEBIT,3,4,91.250000,314.640015,Advance Shipping,0,73,Sporting Goods,Caguas,...,Java Occidental,COMPLETE,1360,73,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class,False
1,TRANSFER,5,4,-249.089996,311.359985,Late Delivery,1,73,Sporting Goods,Caguas,...,RajastÃ¡n,PENDING,1360,73,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class,True
2,CASH,4,4,-247.779999,309.720001,Shipping On Time,0,73,Sporting Goods,San Jose,...,RajastÃ¡n,CLOSED,1360,73,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class,True
3,DEBIT,3,4,22.860001,304.809998,Advance Shipping,0,73,Sporting Goods,Los Angeles,...,Queensland,COMPLETE,1360,73,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class,False
4,PAYMENT,2,4,134.210007,298.250000,Advance Shipping,0,73,Sporting Goods,Caguas,...,Queensland,PENDING_PAYMENT,1360,73,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class,False


## 2. Time-Based Features

Raw timestamps aren't directly usable by most models — the signal is in the parts (seasonality, weekday
effects) and in whether a date falls on a weekend or public holiday, both of which tend to correlate with
staffing levels and, downstream, delivery delays.

In [2]:
order_date = df_fe["order date (DateOrders)"]

df_fe["order_year"] = order_date.dt.year
df_fe["order_quarter"] = order_date.dt.quarter
df_fe["order_month"] = order_date.dt.month
df_fe["order_week"] = order_date.dt.isocalendar().week.astype(int)
df_fe["order_day_of_week"] = order_date.dt.dayofweek  # 0 = Monday
df_fe["order_day_name"] = order_date.dt.day_name()
df_fe["order_is_weekend"] = df_fe["order_day_of_week"].isin([5, 6]).astype(int)

df_fe[["order_year", "order_quarter", "order_month", "order_week",
       "order_day_name", "order_is_weekend"]].head()

,order_year,order_quarter,order_month,order_week,order_day_name,order_is_weekend
0,2018,1,1,5,Wednesday,0
1,2018,1,1,2,Saturday,1
2,2018,1,1,2,Saturday,1
3,2018,1,1,2,Saturday,1
4,2018,1,1,2,Saturday,1


**Holiday flag.** Order Country spans dozens of markets, so a single calendar won't do — a US holiday
flag would mislabel every Indian or Australian order. `holidays` supports per-country calendars, so each
order is checked against its own country's calendar where one is available; countries `holidays` doesn't
cover fall back to 0 rather than a guess.

In [3]:
import holidays
import pycountry

def to_iso_code(country_name: str):
    overrides = {"EE. UU.": "US", "Puerto Rico": "US"}  # dataset-specific spellings / territories
    if country_name in overrides:
        return overrides[country_name]
    try:
        return pycountry.countries.lookup(country_name).alpha_2
    except LookupError:
        return None

order_years = range(df_fe["order_year"].min(), df_fe["order_year"].max() + 1)

country_calendars = {}
for country in df_fe["Order Country"].unique():
    code_ = to_iso_code(country)
    if code_ is None:
        country_calendars[country] = None
        continue
    try:
        country_calendars[country] = holidays.country_holidays(code_, years=order_years)
    except NotImplementedError:
        country_calendars[country] = None

covered = sum(v is not None for v in country_calendars.values())
print(f"Holiday calendars resolved for {covered}/{len(country_calendars)} order countries")

def is_holiday(row):
    cal = country_calendars.get(row["Order Country"])
    if cal is None:
        return 0
    return int(row["order date (DateOrders)"].date() in cal)

df_fe["order_is_holiday"] = df_fe.apply(is_holiday, axis=1)
df_fe["order_is_holiday"].value_counts()

Holiday calendars resolved for 62/164 order countries


order_is_holiday
0    178423
1      2096
Name: count, dtype: int64

## 3. Delivery Performance Features

`Days for shipping (real)` and `Days for shipping (scheduled)` already exist in the cleaned data — the
gap between them is the single strongest signal for late-delivery prediction, so it's built first and
cross-checked against the dataset's own `Late_delivery_risk` flag before anything is layered on top.

In [4]:
df_fe["delivery_time_days"] = df_fe["Days for shipping (real)"]
df_fe["shipping_delay_days"] = (
    df_fe["Days for shipping (real)"] - df_fe["Days for shipment (scheduled)"]
)

df_fe["is_late"] = (df_fe["shipping_delay_days"] > 0).astype(int)
df_fe["is_early"] = (df_fe["shipping_delay_days"] < 0).astype(int)

agreement = (df_fe["is_late"] == df_fe["Late_delivery_risk"]).mean()
print(f"is_late agrees with the dataset's own Late_delivery_risk flag {agreement:.1%} of the time")

df_fe["delivery_performance_category"] = np.select(
    [df_fe["shipping_delay_days"] > 0, df_fe["shipping_delay_days"] < 0],
    ["Late", "Early"],
    default="On Time",
)
df_fe["delivery_performance_category"].value_counts(normalize=True)

is_late agrees with the dataset's own Late_delivery_risk flag 97.5% of the time


delivery_performance_category
Late       0.572793
Early      0.240230
On Time    0.186978
Name: proportion, dtype: float64

In [5]:
# Which shipping method performs best? (full-history aggregate — dashboard use, not per-order leakage-safe)
shipping_mode_perf = (
    df_fe.groupby("Shipping Mode")
    .agg(orders=("Order Id", "count"),
         avg_delay_days=("shipping_delay_days", "mean"),
         late_rate=("is_late", "mean"))
    .sort_values("late_rate")
)
shipping_mode_perf

,orders,avg_delay_days,late_rate
Shipping Mode,,,
Standard Class,107752,-0.004093,0.397682
Same Day,9737,0.478279,0.478279
Second Class,35216,1.990828,0.797308
First Class,27814,1.000000,1.000000


## 4. Sales Features

`Sales` is already the order-item revenue line. Rolling it up to order level gives revenue and order-value
features; a value cutoff at the 90th percentile marks the orders worth flagging for account management.

In [6]:
order_revenue = df_fe.groupby("Order Id")["Sales"].transform("sum")
df_fe["order_total_revenue"] = order_revenue

average_order_value = df_fe.drop_duplicates("Order Id")["order_total_revenue"].mean()
high_value_cutoff = df_fe.drop_duplicates("Order Id")["order_total_revenue"].quantile(0.90)

df_fe["high_value_order_flag"] = (df_fe["order_total_revenue"] >= high_value_cutoff).astype(int)
df_fe["revenue_category"] = pd.qcut(
    df_fe["Sales"], q=[0, 0.25, 0.75, 1.0], labels=["Low", "Medium", "High"]
)

print(f"Average order value: {average_order_value:.2f}")
print(f"High-value cutoff (90th pct): {high_value_cutoff:.2f}")
df_fe[["order_total_revenue", "high_value_order_flag", "revenue_category"]].head()

Average order value: 559.45
High-value cutoff (90th pct): 1049.93


,order_total_revenue,high_value_order_flag,revenue_category
0,327.75,0,High
1,327.75,0,High
2,327.75,0,High
3,327.75,0,High
4,327.75,0,High


## 5. Profit Features

In [7]:
df_fe["profit_margin"] = np.where(
    df_fe["Sales"] > 0, df_fe["Order Profit Per Order"] / df_fe["Sales"], np.nan
)
df_fe["loss_order_flag"] = (df_fe["Order Profit Per Order"] < 0).astype(int)

df_fe["profit_category"] = pd.cut(
    df_fe["profit_margin"],
    bins=[-np.inf, 0, 0.10, 0.25, np.inf],
    labels=["Loss", "Low Margin", "Mid Margin", "High Margin"],
)

category_avg_margin = df_fe.groupby("Category Name")["profit_margin"].transform("mean")
df_fe["category_avg_margin"] = category_avg_margin

df_fe[["profit_margin", "loss_order_flag", "profit_category"]].describe(include="all")

,profit_margin,loss_order_flag,profit_category
count,180519.000000,180519.000000,180519
unique,NaN,NaN,4
top,NaN,NaN,High Margin
freq,NaN,NaN,86295
mean,0.108326,0.187149,NaN
std,0.420594,0.390032,NaN
min,-2.750000,0.000000,NaN
25%,0.062240,0.000000,NaN
50%,0.242512,0.000000,NaN
75%,0.336014,0.000000,NaN


## 6. Customer Features

`customer_order_count`, `customer_total_sales`, and `customer_avg_order_value` describe a customer's full
history — great for CRM segmentation, but leakage for order-time prediction since they include orders that
haven't happened yet relative to any given row. `customer_orders_to_date` is the point-in-time-safe version:
only orders placed *before* the current one count.

In [8]:
df_fe = df_fe.sort_values("order date (DateOrders)")

customer_agg = df_fe.groupby("Customer Id").agg(
    customer_order_count=("Order Id", "nunique"),
    customer_total_sales=("Sales", "sum"),
    customer_late_order_rate=("is_late", "mean"),
)
customer_agg["customer_avg_order_value"] = (
    customer_agg["customer_total_sales"] / customer_agg["customer_order_count"]
)
customer_agg["customer_is_repeat"] = (customer_agg["customer_order_count"] > 1).astype(int)
customer_agg["customer_value_segment"] = pd.qcut(
    customer_agg["customer_total_sales"], q=4, labels=["Bronze", "Silver", "Gold", "Platinum"]
)

df_fe = df_fe.merge(customer_agg, on="Customer Id", how="left")

# Point-in-time-safe count: orders this customer had placed strictly before this one
df_fe["customer_orders_to_date"] = df_fe.groupby("Customer Id").cumcount()

df_fe[["Customer Id", "customer_order_count", "customer_orders_to_date",
       "customer_value_segment", "customer_is_repeat"]].head()

,Customer Id,customer_order_count,customer_orders_to_date,customer_value_segment,customer_is_repeat
0,11599,3,0,Silver,1
1,256,10,0,Platinum,1
2,256,10,1,Platinum,1
3,256,10,2,Platinum,1
4,8827,4,0,Gold,1


## 7. Product Features

In [9]:
product_agg = df_fe.groupby("Product Name").agg(
    product_revenue=("Sales", "sum"),
    product_order_count=("Order Id", "nunique"),
    product_avg_profit=("Order Profit Per Order", "mean"),
)
product_agg["product_sales_rank"] = product_agg["product_revenue"].rank(ascending=False, method="dense")
product_agg["product_demand_category"] = pd.qcut(
    product_agg["product_order_count"], q=[0, 0.33, 0.66, 1.0], labels=["Low", "Medium", "High"], duplicates="drop"
)

price_cutoff = df_fe["Product Price"].quantile(0.75)
df_fe["premium_product_flag"] = (df_fe["Product Price"] >= price_cutoff).astype(int)

df_fe = df_fe.merge(product_agg, on="Product Name", how="left")

product_agg.sort_values("product_revenue", ascending=False).head(10)

,product_revenue,product_order_count,product_avg_profit,product_sales_rank,product_demand_category
Product Name,,,,,
Field & Stream Sportsman 16 Gun Fire Safe,6.929654e+06,15164,43.649106,1.0,High
Perfect Fitness Perfect Rip Deck,4.421143e+06,20359,20.143924,2.0,High
Diamondback Women's Serene Classic Comfort Bi,4.118426e+06,12299,31.135230,3.0,High
Nike Men's Free 5.0+ Running Shoe,3.667633e+06,11092,31.219970,4.0,High
Nike Men's Dri-FIT Victory Golf Polo,3.147800e+06,17869,16.658951,5.0,High
Pelican Sunstream 100 Kayak,3.099845e+06,13727,20.908153,6.0,High
Nike Men's CJ Elite 2 TD Football Cleat,2.891758e+06,18783,14.020625,7.0,High
O'Brien Men's Neoprene Life Vest,2.888994e+06,16623,16.501784,8.0,High
Under Armour Girls' Toddler Spine Surge Runni,1.269083e+06,9825,11.893992,9.0,High


## 8. Shipping Features

The cleaned dataset doesn't carry a separate freight-cost column, so a true "cost per unit" or "cost ratio"
can't be built without inventing numbers. What's available instead: how far off the scheduled promise each
shipment ran, and how that varies by mode — which is what a logistics team actually uses to pick a carrier.

In [10]:
df_fe["shipping_efficiency_score"] = np.where(
    df_fe["Days for shipping (real)"] > 0,
    df_fe["Days for shipment (scheduled)"] / df_fe["Days for shipping (real)"],
    np.nan,
)  # > 1 = beat the promise, < 1 = ran over

shipping_mode_stats = df_fe.groupby("Shipping Mode").agg(
    shipping_mode_avg_delay=("shipping_delay_days", "mean"),
    shipping_mode_late_rate=("is_late", "mean"),
)
df_fe = df_fe.merge(shipping_mode_stats, on="Shipping Mode", how="left")

df_fe[["Shipping Mode", "shipping_efficiency_score", "shipping_mode_avg_delay",
       "shipping_mode_late_rate"]].drop_duplicates("Shipping Mode")

,Shipping Mode,shipping_efficiency_score,shipping_mode_avg_delay,shipping_mode_late_rate
0,Standard Class,2.000000,-0.004093,0.397682
13,Second Class,0.666667,1.990828,0.797308
63,First Class,0.500000,1.000000,1.000000
262,Same Day,0.000000,0.478279,0.478279


## 9. Regional Features

Full-history aggregates, same leakage caveat as the customer features — useful for a regional performance
dashboard, not for a row-level model without a point-in-time recompute.

In [11]:
region_agg = df_fe.groupby("Order Region").agg(
    region_total_revenue=("Sales", "sum"),
    region_total_profit=("Order Profit Per Order", "sum"),
    region_late_rate=("is_late", "mean"),
)
df_fe = df_fe.merge(region_agg, on="Order Region", how="left")

region_agg.sort_values("region_total_revenue", ascending=False)

,region_total_revenue,region_total_profit,region_late_rate
Order Region,,,
Western Europe,5.894381e+06,625446.080548,0.585156
Central America,5.665712e+06,616341.570651,0.572457
South America,2.960881e+06,335154.400817,0.572347
Northern Europe,2.155831e+06,233450.600647,0.564134
Southern Europe,2.047919e+06,230829.229883,0.567278
Oceania,2.016654e+06,201478.020484,0.561096
Southeast Asia,1.932496e+06,211342.819786,0.579830
Caribbean,1.651019e+06,171825.640024,0.558788
West of USA,1.571416e+06,164940.660455,0.565995


## 10. Risk Features

A composite score for "how likely is this order to run late," built only from information known at order
time — the shipping mode picked and the region it's shipping to. It reuses `shipping_mode_late_rate` and
`region_late_rate` from above, so it inherits the same leakage caveat: fine for exploring which orders
*historically* looked risky, not a leakage-safe input to a live model without a point-in-time recompute.

In [12]:
df_fe["high_discount_flag"] = (
    df_fe["Order Item Discount Rate"] >= df_fe["Order Item Discount Rate"].quantile(0.75)
).astype(int)

df_fe["delivery_risk_score"] = (
    0.5 * df_fe["shipping_mode_late_rate"] + 0.5 * df_fe["region_late_rate"]
)
df_fe["high_risk_order_flag"] = (
    df_fe["delivery_risk_score"] >= df_fe["delivery_risk_score"].quantile(0.75)
).astype(int)

df_fe[["delivery_risk_score", "high_risk_order_flag", "high_discount_flag"]].describe()

,delivery_risk_score,high_risk_order_flag,high_discount_flag
count,180519.000000,180519.000000,180519.000000
mean,0.572793,0.259585,0.277782
std,0.119283,0.438408,0.447907
min,0.458486,0.000000,0.000000
25%,0.485014,0.000000,0.000000
50%,0.491364,0.000000,0.000000
75%,0.684882,1.000000,1.000000
max,0.803518,1.000000,1.000000


## 11. Customer Behavior Features

In [13]:
preferred_mode = (
    df_fe.groupby("Customer Id")["Shipping Mode"]
    .agg(lambda modes: modes.value_counts().idxmax())
    .rename("preferred_shipping_mode")
)

tenure = df_fe.groupby("Customer Id")["order date (DateOrders)"].agg(["min", "max"])
tenure_days = (tenure["max"] - tenure["min"]).dt.days.replace(0, np.nan)
purchase_frequency = (customer_agg["customer_order_count"] / tenure_days).rename("purchase_frequency")

basket_size = (
    df_fe.groupby(["Customer Id", "Order Id"])["Order Item Quantity"].sum()
    .groupby("Customer Id").mean()
    .rename("avg_basket_size")
)

behavior_agg = pd.concat([preferred_mode, purchase_frequency, basket_size], axis=1)
df_fe = df_fe.merge(behavior_agg, on="Customer Id", how="left")

behavior_agg.head()

,preferred_shipping_mode,purchase_frequency,avg_basket_size
Customer Id,,,
1,Standard Class,NaN,5.00
2,Standard Class,0.005208,4.75
3,Standard Class,0.009074,6.60
4,Standard Class,0.006504,8.50
5,Second Class,0.022059,6.00


## 12. KPI Features

These are the numbers that go on a dashboard, not into a row-level model — one value each, summarizing the
whole dataset (or sliceable by any dimension above).

In [14]:
kpi_summary = pd.DataFrame([{
    "total_revenue": df_fe["Sales"].sum(),
    "total_profit": df_fe["Order Profit Per Order"].sum(),
    "avg_delivery_time_days": df_fe["delivery_time_days"].mean(),
    "overall_profit_margin": df_fe["Order Profit Per Order"].sum() / df_fe["Sales"].sum(),
    "late_delivery_rate": df_fe["is_late"].mean(),
    "average_order_value": average_order_value,
}])
kpi_summary

,total_revenue,total_profit,avg_delivery_time_days,overall_profit_margin,late_delivery_rate,average_order_value
0,3.678474e+07,3.966903e+06,3.497654,0.107841,0.572793,559.446633


## 13. ML-Ready Feature Set

Two targets are in scope: `is_late` (classification) and `profit_margin` (regression). Features are split
into **point-in-time safe** (known at order time, safe for either model) and **full-history aggregates**
(fine for EDA/dashboards, need a point-in-time recompute — like `customer_orders_to_date` — before they go
into a live model). Low-cardinality categoricals are one-hot encoded; high-cardinality ones (city, country,
category) are frequency encoded instead, to avoid exploding the column count.

In [15]:
high_cardinality_cols = ["Category Name", "Order City", "Order Country"]
for col in high_cardinality_cols:
    freq = df_fe[col].value_counts(normalize=True)
    df_fe[f"{col}_freq_encoded"] = df_fe[col].map(freq)

low_cardinality_cols = ["Shipping Mode", "Customer Segment", "Market", "delivery_performance_category"]
df_fe = pd.get_dummies(df_fe, columns=low_cardinality_cols, drop_first=True)

point_in_time_safe = [
    "order_year", "order_quarter", "order_month", "order_week", "order_day_of_week",
    "order_is_weekend", "order_is_holiday", "shipping_efficiency_score",
    "customer_orders_to_date", "premium_product_flag", "high_discount_flag",
] + [f"{c}_freq_encoded" for c in high_cardinality_cols]

full_history_aggregates = [
    "customer_order_count", "customer_total_sales", "customer_avg_order_value",
    "customer_late_order_rate", "customer_is_repeat", "customer_value_segment",
    "product_revenue", "product_order_count", "product_avg_profit", "product_sales_rank",
    "region_total_revenue", "region_total_profit", "region_late_rate",
    "shipping_mode_avg_delay", "shipping_mode_late_rate", "delivery_risk_score",
    "high_risk_order_flag", "purchase_frequency", "avg_basket_size",
]

print(f"Point-in-time safe features: {len(point_in_time_safe)}")
print(f"Full-history aggregate features (dashboard-only / need recompute): {len(full_history_aggregates)}")
print(f"Final dataset shape: {df_fe.shape}")

df_fe[point_in_time_safe + ["is_late"]].corr(numeric_only=True)["is_late"].sort_values(key=abs, ascending=False)

Point-in-time safe features: 14
Full-history aggregate features (dashboard-only / need recompute): 19
Final dataset shape: (180519, 103)


is_late                       1.000000
shipping_efficiency_score    -0.804083
Category Name_freq_encoded   -0.004608
Order City_freq_encoded      -0.003819
order_is_holiday              0.003285
Order Country_freq_encoded    0.002837
order_week                    0.002790
customer_orders_to_date       0.002766
order_month                   0.002328
high_discount_flag           -0.001218
order_quarter                 0.000993
order_is_weekend             -0.000633
order_year                    0.000460
order_day_of_week            -0.000331
premium_product_flag          0.000262
Name: is_late, dtype: float64

## 14. Feature Validation

Before anything ships: every engineered column gets checked for nulls, impossible values, and whether it
actually agrees with the dataset's own ground truth where one exists.

In [16]:
validation_cols = [
    "delivery_time_days", "shipping_delay_days", "is_late", "profit_margin",
    "order_total_revenue", "customer_orders_to_date", "delivery_risk_score",
]

null_report = df_fe[validation_cols].isnull().mean().rename("pct_null").to_frame()
print("Null check:")
print(null_report)

checks = {
    "delivery_time_days >= 0": (df_fe["delivery_time_days"] >= 0).all(),
    "Order Item Discount Rate within [0, 1]": df_fe["Order Item Discount Rate"].between(0, 1).all(),
    "Order Item Quantity > 0": (df_fe["Order Item Quantity"] > 0).all(),
    "is_late matches Late_delivery_risk (>=95%)": agreement >= 0.95,
}
print("\nBusiness-sense checks:")
for check, passed in checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {check}")

Null check:
                         pct_null
delivery_time_days            0.0
shipping_delay_days           0.0
is_late                       0.0
profit_margin                 0.0
order_total_revenue           0.0
customer_orders_to_date       0.0
delivery_risk_score           0.0

Business-sense checks:
  [PASS] delivery_time_days >= 0
  [PASS] Order Item Discount Rate within [0, 1]
  [PASS] Order Item Quantity > 0
  [PASS] is_late matches Late_delivery_risk (>=95%)


## 15. Export

In [17]:
# df_fe.to_csv("DataCoSupplyChain_FeatureEngineered2.csv", index=False)
# kpi_summary.to_csv("DataCoSupplyChain_KPI_Summary.csv", index=False)
# print("Saved: DataCoSupplyChain_FeatureEngineered.csv")
# print("Saved: DataCoSupplyChain_KPI_Summary.csv")

Saved: DataCoSupplyChain_FeatureEngineered.csv


In [18]:
df_fe.columns

Index(['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Benefit per order', 'Sales per customer', 'Delivery Status',
       'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City',
       ...
       'Shipping Mode_Second Class', 'Shipping Mode_Standard Class',
       'Customer Segment_Corporate', 'Customer Segment_Home Office',
       'Market_Europe', 'Market_Latam', 'Market_Pacific Asia', 'Market_Usca',
       'delivery_performance_category_Late',
       'delivery_performance_category_On Time'],
      dtype='object', length=103)

## 16. Summary

- **Time:** year, quarter, month, week, weekday, weekend flag, and a per-country holiday flag.
- **Delivery:** delivery time, shipping delay, `is_late`/`is_early`, performance category — cross-checked
  against the dataset's own `Late_delivery_risk` flag.
- **Sales & Profit:** order revenue, AOV, high-value flag, profit margin, loss flag, profit category.
- **Customer:** order count, spend, AOV, repeat flag, value segment — with a point-in-time-safe
  `customer_orders_to_date` for modeling.
- **Product:** revenue, sales rank, demand category, premium flag.
- **Shipping:** delay- and efficiency-based proxies (no cost column exists in this dataset to build true
  cost features).
- **Regional & Risk:** region-level revenue/profit/late-rate, a composite delivery risk score, high-discount
  and high-risk flags.
- **Customer Behavior:** preferred shipping mode, purchase frequency, average basket size.
- **KPIs:** a single dashboard-ready summary row.
- **Encoding:** one-hot for low-cardinality categoricals, frequency encoding for high-cardinality ones.
- **Validation:** null checks, impossible-value checks, and agreement against ground truth, before export.
- **Not built:** inventory features (no inventory table in this dataset).
- Exported `DataCoSupplyChain_FeatureEngineered.csv` and `DataCoSupplyChain_KPI_Summary.csv`, ready for
  modeling and dashboarding respectively.